# Fase 3 — Data Preparation (CRISP-DM)
## Sistema Inteligente de Recomendación de Películas — MovieLens 25M

Notebook `02_Data_Sampling_and_Cleaning.ipynb` — **CRISP-DM Fase 3 (Data Preparation)**.

Este notebook continúa el pipeline después de `01_Business_Understanding_and_EDA.ipynb` (Fases 1 y 2). Toma los hallazgos del EDA y produce dos datasets listos para modelar a partir de los 25 M de calificaciones originales:

- una **muestra principal estratificada al 60 %** para baseline, SVD y NMF,
- una **submuestra al 10 %** reservada para KNN como benchmark metodológico.

**Decisiones incorporadas en esta versión:**
- Muestreo **estratificado por tier** para preservar el mix de usuarios casuales, regulares y power users.
- **Sin filtro global de cold-start** sobre el catálogo: el long tail se conserva y el cold-start se mide luego en evaluación.
- `timestamp` se preserva para habilitar **split temporal por usuario** en el notebook 03.

El siguiente notebook (`03_ML_Baseline_AutoML.ipynb`) consume los parquets producidos aquí.


## 1. Reproducibilidad, rutas y chequeo de hardware

Antes de tocar el dataset definimos:
- Semilla global `SEED=42` para `random`, `numpy`, `PYTHONHASHSEED` y cualquier `sample(random_state=...)` posterior.
- Rutas **relativas a la raíz del proyecto** (no rutas absolutas de Deepnote): el notebook se autolocaliza detectando si está dentro de `notebooks/`.
- Verificación de RAM/CPU con `psutil`: el paso crítico (lectura de `ratings.csv` con pandas) consume ~2.5 GB; si la RAM disponible es inferior caemos al fallback Polars lazy.


In [ ]:
import os
import gc
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import psutil
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

MAIN_SAMPLE_FRACTION = 0.60
KNN_SAMPLE_FRACTION  = 0.10
N_TIERS              = 3

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (11, 5)

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
DATA_RAW_DIR = ROOT / 'data' / 'ml-25m'
DATA_INT_DIR = ROOT / 'data' / 'intermediate'
DATA_INT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_RAW_DIR.exists(), f'No se encuentra {DATA_RAW_DIR}. Descarga MovieLens 25M en data/ml-25m/.'

vm = psutil.virtual_memory()
ram_total_gb = vm.total / (1024 ** 3)
ram_avail_gb = vm.available / (1024 ** 3)
cpus_logical = psutil.cpu_count(logical=True)
cpus_physical = psutil.cpu_count(logical=False)

print(f'Raíz del proyecto : {ROOT}')
print(f'Datos crudos      : {DATA_RAW_DIR}')
print(f'Datos intermedios : {DATA_INT_DIR}')
print('-' * 60)
print(f'Python            : {os.sys.version.split()[0]}')
print(f'NumPy             : {np.__version__}')
print(f'pandas            : {pd.__version__}')
print(f'Polars            : {pl.__version__}')
print('-' * 60)
print(f'CPU               : {cpus_physical} físicos / {cpus_logical} lógicos')
print(f'RAM total         : {ram_total_gb:5.1f} GB')
print(f'RAM disponible    : {ram_avail_gb:5.1f} GB')

USE_POLARS_FALLBACK = ram_avail_gb < 4
print()
if USE_POLARS_FALLBACK:
    print('ADVERTENCIA: RAM libre < 4 GB. Se usará Polars lazy en lugar de pandas para la carga del ratings.csv.')
else:
    print('OK: RAM suficiente para la estrategia pandas directa.')

RATINGS_CSV       = DATA_RAW_DIR / 'ratings.csv'
MOVIES_CSV        = DATA_RAW_DIR / 'movies.csv'
GENOME_SCORES_CSV = DATA_RAW_DIR / 'genome-scores.csv'
GENOME_TAGS_CSV   = DATA_RAW_DIR / 'genome-tags.csv'
for _p in (RATINGS_CSV, MOVIES_CSV, GENOME_SCORES_CSV, GENOME_TAGS_CSV):
    assert _p.exists(), f'Falta: {_p}'
print('Rutas a los 4 CSV verificadas.')


## 2. Estrategia de preparación y pasos

### 2.1 Estrategia

**Meta:** producir dos datasets derivados del MovieLens 25M que sean:

1. **Representativos** del comportamiento de usuarios vía muestreo estratificado por actividad.
2. **Escalables** para entrenar modelos clásicos sobre hardware local o estaciones medianas.
3. **Compatibles con evaluación temporal**, preservando `timestamp` y el catálogo de cola larga.

**Artefactos objetivo:**
- `ratings_prepared_60pct.parquet` + catálogo asociado para Baseline / SVD / NMF.
- `ratings_knn_10pct.parquet` + catálogo asociado para KNN item-based.

**Pasos:**

| Paso | Operación | Herramienta |
|---|---|---|
| 2.1 | Carga `ratings.csv` con `usecols` (ahorro de RAM). | pandas o Polars fallback |
| 2.2 | Calcular actividad por usuario y asignar **3 tiers** con `pd.qcut`. | pandas |
| 2.3 | Muestrear usuarios por tier para dos fracciones (`60 %` y `10 %`). | pandas `.groupby().sample()` |
| 2.4 | Filtrar `ratings` por los usuarios seleccionados y validar integridad. | pandas `.isin()` |
| 2.5 | Persistir ratings y catálogos asociados en Parquet. | `to_parquet` / Polars |
| 2.6 | Validar representatividad y cobertura del catálogo. | pandas / métricas resumen |


### 2.1 Carga eficiente del CSV
Seleccionamos sólo las 4 columnas que necesitamos (sin el `timestamp` temporalmente descartamos nada, lo conservamos para posibles modelos temporales en fases futuras).

In [ ]:
print(f'Cargando ratings.csv (~{RATINGS_CSV.stat().st_size / (1024**2):.0f} MB)...')

if USE_POLARS_FALLBACK:
    df_ratings = (
        pl.scan_csv(
            RATINGS_CSV,
            schema_overrides={'userId': pl.Int32, 'movieId': pl.Int32,
                              'rating': pl.Float32, 'timestamp': pl.Int64},
        )
        .select(['userId', 'movieId', 'rating', 'timestamp'])
        .collect(streaming=True)
        .to_pandas()
    )
    print(f'   Cargado con Polars lazy (streaming)')
else:
    df_ratings = pd.read_csv(
        RATINGS_CSV,
        usecols=['userId', 'movieId', 'rating', 'timestamp'],
        dtype={'userId': 'int32', 'movieId': 'int32',
               'rating': 'float32', 'timestamp': 'int64'},
    )
    print(f'   Cargado con pandas directo')

print(f'   Filas           : {len(df_ratings):>12,}')
print(f'   Usuarios únicos : {df_ratings["userId"].nunique():>12,}')
print(f'   Películas único : {df_ratings["movieId"].nunique():>12,}')
print(f'   Memoria usada   : {df_ratings.memory_usage(deep=True).sum() / (1024**2):>8.1f} MB')

# Guardar estadísticas poblacionales para la validación posterior (sección 4)
POP_N_RATINGS = len(df_ratings)
POP_STATS = {
    'users': df_ratings['userId'].nunique(),
    'items': df_ratings['movieId'].nunique(),
    'mean':  float(df_ratings['rating'].mean()),
    'std':   float(df_ratings['rating'].std()),
}
print(f'   POP_STATS       : {POP_STATS}')

### 2.2 Tiers de actividad de usuario

Dividimos a los usuarios en **3 tiers** según el número total de ratings emitidos, usando **terciles** (`qcut` con q=3):
- `Casual`: tercil inferior.
- `Regular`: tercil medio.
- `PowerUser`: tercil superior.

**Por qué terciles:** la distribución es tan sesgada que un único corte alto agrupa demasiado a los usuarios poco activos. Con terciles cada estrato conserva masa suficiente y permite muestrear tanto el `60 %` principal como el `10 %` de KNN sin distorsionar proporciones.


In [ ]:
user_counts = df_ratings['userId'].value_counts().rename_axis('userId').reset_index(name='n_ratings')
user_counts['tier'] = pd.qcut(
    user_counts['n_ratings'],
    q=N_TIERS,
    labels=['Casual', 'Regular', 'PowerUser'],
)

tier_summary = user_counts.groupby('tier', observed=True)['n_ratings'].agg(['count', 'min', 'median', 'max']).rename(
    columns={'count': 'n_usuarios'}
)
print('Distribución de tiers (población completa):')
print(tier_summary)

# Guardamos para validación posterior
POP_TIER_COUNTS = user_counts['tier'].value_counts(normalize=True).sort_index()
print(f'\nProporción poblacional por tier:\n{POP_TIER_COUNTS.round(4).to_dict()}')

### 2.3 Muestreo estratificado para dos vistas del dataset

Extraemos usuarios **dentro de cada tier** con `random_state=SEED` para reproducibilidad. Construimos dos vistas:

- **Muestra principal (`60 %`)**: se usa para los modelos productivos del notebook 03.
- **Muestra KNN (`10 %`)**: conserva el mismo criterio de estratificación, pero limita el costo del modelo memory-based.


In [ ]:
def stratified_user_sample(df, frac, seed=SEED):
    sampled = (
        df.groupby('tier', group_keys=False, observed=True)
        .apply(lambda g: g.sample(frac=frac, random_state=seed))
        .reset_index(drop=True)
    )
    tier_counts = sampled['tier'].value_counts(normalize=True).sort_index()
    diff = (tier_counts - POP_TIER_COUNTS).abs().max()
    print(f'Usuarios seleccionados ({frac*100:.0f}%): {len(sampled):,}')
    print(sampled.groupby('tier', observed=True).size().rename('n_usuarios_muestra'))
    print(f'Máx. diferencia proporcional vs población: {diff:.4f}')
    assert diff < 0.01, 'El muestreo estratificado debería preservar proporciones con < 1 % de desvío.'
    return sampled, tier_counts

sampled_users_main_df, MAIN_TIER_COUNTS = stratified_user_sample(user_counts, MAIN_SAMPLE_FRACTION)
print('-' * 60)
sampled_users_knn_df, KNN_TIER_COUNTS = stratified_user_sample(user_counts, KNN_SAMPLE_FRACTION)

sampled_user_ids_main = set(sampled_users_main_df['userId'].tolist())
sampled_user_ids_knn  = set(sampled_users_knn_df['userId'].tolist())


### 2.4 Filtrar ratings por usuarios muestreados y validar integridad


In [ ]:
def build_sample(df, sampled_user_ids, sample_name):
    sample_df = df[df['userId'].isin(sampled_user_ids)].copy()
    print(f'[{sample_name}] Ratings retenidos : {len(sample_df):>10,}  (~{100*len(sample_df)/len(df):.2f} % de la población)')
    print(f'[{sample_name}] Usuarios únicos   : {sample_df["userId"].nunique():>10,}')
    print(f'[{sample_name}] Películas únicas  : {sample_df["movieId"].nunique():>10,}')

    assert sample_df['userId'].notnull().all()
    assert sample_df['movieId'].notnull().all()
    assert sample_df['rating'].between(0.5, 5.0).all()
    assert not sample_df.duplicated(subset=['userId', 'movieId']).any(), 'No puede haber (user,movie) duplicados.'

    per_user = sample_df.groupby('userId').size()
    assert per_user.min() >= 1, f'[{sample_name}] Hay usuarios sin ratings tras el filtrado.'

    return sample_df

ratings_main = build_sample(df_ratings, sampled_user_ids_main, 'MAIN 60%')
ratings_knn  = build_sample(df_ratings, sampled_user_ids_knn,  'KNN 10%')

del df_ratings
gc.collect()
print(f'Memoria libre ahora: {psutil.virtual_memory().available / 1024**3:.1f} GB')


### 2.5 Persistencia en formato Parquet

Persistimos los dos datasets de ratings sin filtro global de popularidad. La evaluación de *cold-start* se hará más adelante con split temporal, en vez de eliminar la cola larga desde preparación.


In [ ]:
RATINGS_MAIN_PARQUET = DATA_INT_DIR / 'ratings_prepared_60pct.parquet'
RATINGS_KNN_PARQUET  = DATA_INT_DIR / 'ratings_knn_10pct.parquet'

ratings_main.to_parquet(RATINGS_MAIN_PARQUET, index=False, compression='snappy')
ratings_knn.to_parquet(RATINGS_KNN_PARQUET, index=False, compression='snappy')

for label, frame, path in (
    ('MAIN 60%', ratings_main, RATINGS_MAIN_PARQUET),
    ('KNN 10%', ratings_knn, RATINGS_KNN_PARQUET),
):
    csv_equiv_mb = len(frame) * 30 / (1024 ** 2)
    parquet_mb = path.stat().st_size / (1024 ** 2)
    print(f'[{label}] Ratings guardados en : {path.name}')
    print(f'[{label}] Tamaño Parquet       : {parquet_mb:5.2f} MB')
    print(f'[{label}] Tamaño estimado CSV  : {csv_equiv_mb:5.2f} MB  (ratio ≈ {csv_equiv_mb/parquet_mb:.1f}×)')
    print(f'[{label}] Catálogo retenido    : {frame["movieId"].nunique():,} películas')

VALID_MOVIE_IDS_MAIN = set(ratings_main['movieId'].unique().tolist())
VALID_MOVIE_IDS_KNN  = set(ratings_knn['movieId'].unique().tolist())


### 2.6 Persistencia en formato Parquet

**Por qué Parquet y no CSV:**
- **Compresión columnar** (Snappy): ~6× más pequeño que CSV.
- **Tipado preservado**: `int32`, `float32` se leen directo sin parseo de strings.
- **Lectura selectiva**: los notebooks siguientes pueden leer sólo `['userId', 'movieId']` en segundos.
- **Estándar de facto** en el stack pandas/Polars/PyArrow.

La persistencia de ratings ya se realizó en la celda anterior con los artefactos:

- `ratings_prepared_60pct.parquet`
- `ratings_knn_10pct.parquet`

A continuación sincronizamos los catálogos y `genome-scores` para cada vista del dataset.


## 3. Sincronización de metadatos

Para mantener la coherencia entre tablas, filtramos `movies.csv` y `genome-scores.csv` a los `movieId` presentes en cada dataset preparado.

- La vista principal (`60 %`) alimenta Baseline, SVD, NMF y la app.
- La vista `10 %` se deja lista para el benchmark KNN.
- `genome-tags.csv` se copia completo una sola vez porque es un catálogo pequeño.


In [ ]:
movies_raw = pd.read_csv(MOVIES_CSV, dtype={'movieId': 'int32'})

def persist_catalog(valid_movie_ids, ratings_name, movies_name, genome_name):
    movies_filtered = movies_raw[movies_raw['movieId'].isin(valid_movie_ids)].reset_index(drop=True)
    movies_path = DATA_INT_DIR / movies_name
    movies_filtered.to_parquet(movies_path, index=False, compression='snappy')
    print(f'{ratings_name}: {len(movies_filtered):,} películas → {movies_path.name}')

    genome_filtered = (
        pl.scan_csv(
            GENOME_SCORES_CSV,
            schema_overrides={'movieId': pl.Int32, 'tagId': pl.Int32, 'relevance': pl.Float32},
        )
        .filter(pl.col('movieId').is_in(list(valid_movie_ids)))
        .collect(streaming=True)
    )
    genome_path = DATA_INT_DIR / genome_name
    genome_filtered.write_parquet(genome_path, compression='snappy')
    print(f'{ratings_name}: {genome_filtered.height:,} pares (movie, tag) → {genome_path.name}')
    return movies_filtered, genome_filtered.height

movies_main, genome_main_rows = persist_catalog(
    VALID_MOVIE_IDS_MAIN,
    'MAIN 60%',
    'movies_prepared_60pct.parquet',
    'genome_scores_prepared_60pct.parquet',
)
movies_knn, genome_knn_rows = persist_catalog(
    VALID_MOVIE_IDS_KNN,
    'KNN 10%',
    'movies_knn_10pct.parquet',
    'genome_scores_knn_10pct.parquet',
)

tags_cat = pd.read_csv(GENOME_TAGS_CSV, dtype={'tagId': 'int32'})
tags_out = DATA_INT_DIR / 'genome_tags.parquet'
tags_cat.to_parquet(tags_out, index=False, compression='snappy')
print(f'GENOME TAGS: {len(tags_cat):,} tags → {tags_out.name}')

del movies_raw, tags_cat
gc.collect()


## 4. Validación de la muestra — representatividad y cobertura

Un muestreo que distorsione la distribución original invalida cualquier modelo entrenado sobre él. Comparamos la vista principal (`60 %`) contra la población en tres dimensiones:

1. Distribución de `rating`.
2. Distribución de `género`.
3. Distribución de `tiers` de usuario.

Usamos la **métrica L1** (suma de |Δ|) sobre distribuciones normalizadas. Además reportamos cobertura del catálogo retenido, ya sin el recorte artificial de cold-start.


In [ ]:
# 4.1 — Distribución de rating: población (lazy) vs muestra principal
pop_rating_dist = (
    pl.scan_csv(
        RATINGS_CSV,
        schema_overrides={'userId': pl.Int32, 'movieId': pl.Int32, 'rating': pl.Float32, 'timestamp': pl.Int64},
    )
    .group_by('rating').agg(pl.len().alias('n'))
    .sort('rating')
    .collect()
    .to_pandas()
)
pop_rating_dist['pct'] = pop_rating_dist['n'] / pop_rating_dist['n'].sum()

main_rating_dist = ratings_main.groupby('rating').size().reset_index(name='n')
main_rating_dist['pct'] = main_rating_dist['n'] / main_rating_dist['n'].sum()

rating_cmp = pop_rating_dist[['rating', 'pct']].rename(columns={'pct': 'pop'}).merge(
    main_rating_dist[['rating', 'pct']].rename(columns={'pct': 'muestra'}),
    on='rating', how='left'
)
rating_cmp['delta'] = (rating_cmp['pop'] - rating_cmp['muestra']).abs()

l1_rating = rating_cmp['delta'].sum()
print('Comparación de distribución de rating (población vs muestra principal):')
print(rating_cmp.round(4))
print()
print(f'L1 distance: {l1_rating:.4f}   (excelente si < 0.02)')


In [ ]:
# 4.2 — Distribución de géneros: catálogo completo vs muestra principal
movies_pop = pd.read_csv(MOVIES_CSV, dtype={'movieId': 'int32'})

def genre_distribution(df_movies):
    exploded = df_movies.assign(g=df_movies['genres'].str.split('|')).explode('g')
    return exploded['g'].value_counts(normalize=True).rename_axis('g').reset_index(name='pct')

pop_genre_dist = genre_distribution(movies_pop)
main_genre_dist = genre_distribution(movies_main)

genre_cmp = pop_genre_dist.rename(columns={'pct': 'pop'}).merge(
    main_genre_dist.rename(columns={'pct': 'muestra'}),
    on='g', how='left'
).fillna({'muestra': 0.0})
genre_cmp['delta'] = (genre_cmp['pop'] - genre_cmp['muestra']).abs()
genre_cmp = genre_cmp.sort_values('pop', ascending=False)

l1_genre = genre_cmp['delta'].sum()
print('Comparación de distribución de género (población vs muestra principal):')
print(genre_cmp.head(15).round(4))
print()
print(f'L1 distance género: {l1_genre:.4f}')


In [ ]:
# 4.3 — Visualización comparativa: rating + género + tiers (muestra principal)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = np.arange(len(rating_cmp))
w = 0.4
axes[0].bar(x - w/2, rating_cmp['pop'],     w, label='Población', color='#3182bd')
axes[0].bar(x + w/2, rating_cmp['muestra'], w, label='Muestra 60%', color='#e6550d')
axes[0].set_xticks(x); axes[0].set_xticklabels(rating_cmp['rating'].astype(str))
axes[0].set_title(f'Distribución de rating (L1={l1_rating:.4f})')
axes[0].set_xlabel('Rating'); axes[0].set_ylabel('Proporción')
axes[0].legend()

top = genre_cmp.head(12).reset_index(drop=True)
x = np.arange(len(top))
axes[1].bar(x - w/2, top['pop'],     w, label='Población', color='#3182bd')
axes[1].bar(x + w/2, top['muestra'], w, label='Muestra 60%', color='#e6550d')
axes[1].set_xticks(x); axes[1].set_xticklabels(top['g'], rotation=45, ha='right')
axes[1].set_title(f'Top-12 géneros (L1 total={l1_genre:.4f})')
axes[1].set_ylabel('Proporción del catálogo')
axes[1].legend()

tiers = list(POP_TIER_COUNTS.index)
x = np.arange(len(tiers))
axes[2].bar(x - w/2, POP_TIER_COUNTS.values,  w, label='Población', color='#3182bd')
axes[2].bar(x + w/2, MAIN_TIER_COUNTS.values, w, label='Muestra 60%', color='#e6550d')
axes[2].set_xticks(x); axes[2].set_xticklabels(tiers)
axes[2].set_title('Proporción de usuarios por tier')
axes[2].set_ylabel('Proporción')
axes[2].legend()

plt.tight_layout(); plt.show()


**Lectura.** La muestra principal al `60 %` conserva la mezcla de usuarios por tier y mantiene mucho mejor la cobertura de catálogo que la versión antigua con cold-start dentro del sample. La matriz sigue siendo sparse, pero ahora el long tail se preserva para evaluación y descubrimiento.


## 5. Métricas finales de las muestras y cobertura del catálogo


In [ ]:
def summarize_sample(name, frame):
    n_users = frame['userId'].nunique()
    n_movies = frame['movieId'].nunique()
    n_ratings = len(frame)
    total_cells = n_users * n_movies
    density = n_ratings / total_cells * 100
    sparsity = 100 - density

    summary_rows = [
        ('Usuarios',  n_users, POP_STATS['users']),
        ('Películas', n_movies, POP_STATS['items']),
        ('Ratings',   n_ratings, POP_N_RATINGS),
    ]
    summary_df = pd.DataFrame(summary_rows, columns=['Métrica', 'Muestra', 'Población'])
    summary_df['% retenido'] = (100 * summary_df['Muestra'] / summary_df['Población']).round(2)
    print()
    print(f'===== {name} =====')
    print(summary_df.to_string(index=False))
    print(f'Rating medio     (muestra / población) : {frame["rating"].mean():.4f} / {POP_STATS["mean"]:.4f}')
    print(f'Rating std       (muestra / población) : {frame["rating"].std():.4f} / {POP_STATS["std"]:.4f}')
    print(f'Ratings/usuario (promedio muestra)     : {n_ratings / n_users:.1f}')
    print(f'Ratings/película (promedio muestra)    : {n_ratings / n_movies:.1f}')
    print(f'Sparsity muestra                      : {sparsity:6.3f} %')
    print(f'Density  muestra                      : {density:6.3f} %')

summarize_sample('MUESTRA PRINCIPAL 60%', ratings_main)
summarize_sample('MUESTRA KNN 10%', ratings_knn)
print()
print('Cobertura de catálogo adicional que aporta la muestra principal sobre KNN:')
print(f'Películas exclusivas del 60% frente al 10%: {len(VALID_MOVIE_IDS_MAIN - VALID_MOVIE_IDS_KNN):,}')
print(f'Genome rows 60% / 10%                : {genome_main_rows:,} / {genome_knn_rows:,}')


**Lectura.** La vista principal al `60 %` se usa para modelado y deployment porque retiene mucha más cobertura del catálogo. La vista `10 %` se reserva a KNN para mantener factible el costo de similitud item-item sin sacrificar la comparación metodológica.


## 6. Artefactos generados y handoff a la siguiente fase

Listamos los ficheros producidos. Desde este punto, los notebooks 02-03 consumen exclusivamente `data/intermediate/` y `data/ml-25m/` (nunca escriben).

In [ ]:
print(f'Artefactos en {DATA_INT_DIR}:')
print('-' * 70)
for p in sorted(DATA_INT_DIR.glob('*.parquet')):
    size_mb = p.stat().st_size / (1024 ** 2)
    rel = p.relative_to(ROOT)
    print(f'   {str(rel):<55} {size_mb:>7.2f} MB')

print('\nSmoke-test de lectura:')
for p in sorted(DATA_INT_DIR.glob('*.parquet')):
    df = pd.read_parquet(p)
    print(f'   {p.name:<35} shape={df.shape}   cols={list(df.columns)}')

# Limpieza final de memoria
del df_final
gc.collect()
print(f'\nMemoria libre al final: {psutil.virtual_memory().available / 1024**3:.1f} GB')

## 7. Conclusiones

| # | Resultado | Implicación para la siguiente fase |
|---|---|---|
| 1 | Sample de ~1.2 M ratings producido con muestreo estratificado al 5 %. | El notebook 01 ya hizo el EDA sobre los 25 M; los notebooks 03-05 consumen este sample. |
| 2 | Cold-start ≥ 20 votos aplicado → ~6 k películas supervivientes. | El testset Surprise no contendrá películas nunca vistas en train. |
| 3 | Representatividad validada: L1(rating) < 0.02, proporciones de tier < 1 % de desvío. | Los modelos entrenados sobre este sample generalizan a la población. |
| 4 | Sparsity ≈ 98.4 % (densa ≈ 1.6 %). | Aún obliga a modelos latentes (SVD/ALS/NMF/NCF); justifica el enfoque de la Fase 4. |
| 5 | Todos los metadatos sincronizados (movies + genome-scores + genome-tags). | El notebook 05 (RAG) puede cruzar `movieId` con tags semánticos sin valores nulos. |
| 6 | Reproducibilidad garantizada por `SEED=42` en todos los `sample()`. | Cualquier ejecución produce exactamente los mismos parquets. |

### Próximo paso
Ejecutar `03_ML_Baseline_AutoML.ipynb` (CRISP-DM Fase 4 — Modeling + Fase 5 — Evaluation) sobre el dataset **original de 25 M**, usando Polars lazy para evitar cargar los datos a RAM.

Los parquets generados aquí son el único insumo que consumen los notebooks 03-05 a partir de este punto.